# Session 8 — The lemonade AMM

This is the **practice notebook**. For each idea: run the **Demo** (it already works), read **Keep this straight**, then fill **YOUR TURN** (`???`). `???` will not run until you replace it. There is no answer key here.

Missed class, or a word is still fuzzy? Re-run the Demo, then open `site/handbook.html`. You do not need to read it before we start.

An **Automated Market Maker** is a fishbowl with two kinds of cash.
The rule: **the product of the two piles stays the same.**

\(x \times y = k\)

Nobody has to sit there yelling buy/sell. The bowl *is* the market.



## Words we use today

Read this once. If a later cell uses the word, come back here.

| Word | In this class it means |
|---|---|
| **AMM** | A fishbowl with two piles. The bowl *is* the market. No human market-maker. |
| **Constant product** | x × y = k. After a trade the product stays the same. |
| **Spot price** | y / x — CLASSUSD per one tiny extra TEAM, *right now*. |
| **Average price** | What *you* actually paid for this chunk (paid ÷ team_out). |
| **Slippage** | You moved the bowl, so you paid worse than the starting spot. |
| **LP** | Liquidity provider: the person who *filled* the jar. |
| **Impermanent loss** | Jar vs just holding the two stacks. A snapshot gap, not a fee. |



## 1. Buy TEAM, slide along the curve

Start: 100 TEAM, 100 CLASSUSD. \(k = 10{,}000\). Spot = 1.

Buy 10 TEAM → 90 left → dollars become \(10{,}000/90 = 111.11\).
You paid **11.11**. Average = 1.11. New spot = 1.23.

You paid 1.11 on average. The *next* sip is 1.23. That gap is **slippage**.

**Demo.** Quiz 2 numbers — you are allowed to see these in class.



In [ ]:
# Demo — constant product, no fees. Quiz 2 numbers.

def buy_team(x, y, team_out):
    k = x * y
    new_x = x - team_out
    if new_x <= 0:
        raise ValueError("cannot buy the whole bowl")
    new_y = k / new_x
    paid = new_y - y
    spot = new_y / new_x
    avg = paid / team_out
    return new_x, new_y, paid, spot, avg


nx, ny, paid, spot, avg = buy_team(100, 100, 10)
print("new_x", nx, "new_y", round(ny, 4))
print("paid", round(paid, 4), "avg", round(avg, 4), "new spot", round(spot, 4))
print("start spot was", 100 / 100, "-- you moved the bowl")



**What you just saw.** x × y stays k. Spot is the *next* sip. Average is what *this* gulp cost. The gap is slippage.

### Keep this straight

**If you remember one thing.** x × y stays k. Big buys move the price against you. That extra pain is slippage.

**Common mix-up.** Treating spot and average as the same number. Spot is the *next* sip. Average is what *this* gulp cost.

**Why Eagecoin cares.** Your class pool is EAGE / CLASSUSD. Quiz 2 will ask 100/100 buy 10: paid 11.11, new spot 1.23.



**Picture.** Same Quiz 2 numbers. You slide *along* the curve; the product stays 10,000.



In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.4, 3.9),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Picture: xy = k. Buying TEAM slides you up the curve — you move the price.
import numpy as np

k = 10_000
x0, y0 = 100.0, 100.0
x1, y1 = 90.0, k / 90.0
xs = np.linspace(40, 180, 240)
ys = k / xs

fig, ax = plt.subplots()
ax.plot(xs, ys, color="#231f1b", lw=2, label="x × y = 10,000")
ax.scatter([x0, x1], [y0, y1], s=70, zorder=3, color="#c45c26")
ax.annotate("start (100, 100)\nspot = 1.00", (x0, y0), xytext=(118, 70),
            arrowprops=dict(arrowstyle="->", color="#666"), fontsize=10)
ax.annotate("after buy 10 TEAM\n(90, 111.11)\nspot = 1.23", (x1, y1), xytext=(45, 145),
            arrowprops=dict(arrowstyle="->", color="#666"), fontsize=10)
ax.set_xlabel("TEAM in the bowl (x)")
ax.set_ylabel("CLASSUSD in the bowl (y)")
ax.set_title("AMM: you paid 11.11; the next sip costs 1.23 — that gap is slippage")
ax.set_xlim(40, 180)
ax.set_ylim(40, 180)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()



## YOUR TURN 1 — L2 bowl, then IL vs hold

L2: 200 TEAM, 100 CLASSUSD, buy 40.
Then compare the jar to just holding the two stacks at the new spot.



In [ ]:
# Reuse buy_team from the Demo.

nx, ny, paid, spot, avg = buy_team(???, ???, ???)  # 200, 100, 40
print("L2", nx, ny, paid, spot, avg)

# IL vs hold after that trade (start was 200 TEAM + 100 CLASSUSD).
pool_value = ???          # nx * spot + ny
hold_value = ???          # 200 * spot + 100
print("withdraw jar", pool_value)
print("just hold", hold_value)
print("gap (IL snapshot)", hold_value - pool_value)



## 2. Trader reverse vs LP “I should have held”

Zero fees: if the *trader* sells the same 10 TEAM back, the bowl returns
to 100/100. The **LP** who filled the jar still had a gap *during* the
trade vs holding stacks. Do not mix those two stories.

**Demo.** Unwind the Quiz 2 buy, then print the LP gap at the *mid* point.



In [ ]:
# Demo -- trader can unwind at 0 fees; LP vs-hold is a different object.

def sell_team(x, y, team_in):
    k = x * y
    new_x = x + team_in
    new_y = k / new_x
    received = y - new_y
    return new_x, new_y, received


mid_x, mid_y, paid, spot, avg = buy_team(100, 100, 10)
print("after buy", mid_x, round(mid_y, 4), "paid", round(paid, 4))

# LP snapshot WHILE the bowl is TEAM-poor (before unwind).
lp_jar = mid_x * spot + mid_y
lp_hold = 100 * spot + 100
print("LP jar", lp_jar, "hold stacks", lp_hold, "gap", round(lp_hold - lp_jar, 4))

back_x, back_y, got = sell_team(mid_x, mid_y, 10)
print("trader reverse ->", back_x, round(back_y, 4), "CLASSUSD back", round(got, 4))
print("trader round-trip works at 0 fees. IL is the LP story, not a fee.")



**What you just saw.** IL is jar vs hold — a snapshot gap, not a trading fee. Zero-fee traders can unwind; LPs still felt the move.

### Keep this straight

**If you remember one thing.** The trader can reverse at 0 fees. The LP still had a gap *during* the trade vs holding stacks.

**Common mix-up.** Calling the IL gap a fee you were charged. There are no fees on the class curve.

**Why Eagecoin cares.** Your class pool is EAGE / CLASSUSD. A classmate not on your team must swap. Write starting reserves and the implied price.



## YOUR TURN 2 — *your* A4 piles (type numbers from your pool, not invented prices)



In [ ]:
# Type YOUR starting reserves. If the pool is not live, say so and use a labeled example.

x = ???          # TEAM in YOUR bowl
y = ???          # CLASSUSD in YOUR bowl
team_out = ???   # a classmate swap size

nx, ny, paid, spot, avg = buy_team(x, y, team_out)
print("your bowl after swap", nx, ny)
print("paid", paid, "avg", avg, "new spot", spot)
print("Would you hold TEAM overnight vs CLASSUSD? One sentence about USE.")



## Exit

Starting reserves, implied price, one external swap, would you hold TEAM
overnight vs CLASSUSD? That paragraph is A4.

L2 (`notebooks/labs/L2_amm.ipynb`) — same bowl, with a plot to turn in.



## Check you understand (say it out loud)

1. 100/100 bowl, buy 10 TEAM. What do you pay, and what is the new spot?
2. Why is average price lower than the new spot after you buy?
3. IL is the gap between ____ and ____. Is it a fee you were charged?

If an answer is a slogan (“decentralized,” “web3,” “the future of money”),
you do not understand it yet. Re-run the Demo cells. Still stuck? The
**concept handbook** (`site/handbook.html`) is the missed-class review —
cafeteria, key, and lemonade pictures, not a second lecture.

